# 01 · 학습 데이터와 누출 점검

지도 미세조정(Supervised Fine-Tuning, SFT) 준비에서 형식·출력 계약·사례 단위 분할을 확인합니다. 약 15분.

모델 다운로드·API 키·GPU 없이 실행합니다. 합성 자료만 사용하며, 실제 LLM 학습·성능 검증과 구분합니다. 예측 → 실행 → 변수 변경 → 근거 기록의 순서로 진행합니다.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from course_labs.adaptation import *
OUT = ROOT / 'output' / 'validation' / 'adaptation-labs'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'NumPy:', np.__version__)
assert sys.version_info[:2] == (3, 11), '공통 환경 Python 3.11 커널을 선택하세요.'


## 합성 사례 48건

24개의 사건마다 표현이 다른 두 문의를 구성합니다. 개별 문장과 사건 ID의 차이를 확인하세요.

In [ ]:
data = examples()
print(len(data)); display(data[:2])

## 정상 데이터 점검

형식 통과는 내용의 정확성이나 사용 권한의 승인을 뜻하지 않습니다.

In [ ]:
report = audit(data)
display(report)
assert not report['issues'] and not report['duplicate_ids']

## 중복의 주입

복사된 사례 한 건을 추가했을 때 ID 중복과 내용 중복은 각각 어떻게 잡힐까요?

In [ ]:
bad = [dict(r) for r in data] + [dict(data[0])]
display(audit(bad))
assert audit(bad)['duplicate_content'] == 1

## 출력 계약의 위반

JSON 구문만 맞는 것과 필요한 필드가 맞는 것은 다릅니다.

In [ ]:
bad[1]['response'] = '{broken}'
bad[2]['response'] = json.dumps({'answer': 'ok'})
display(audit(bad)['issues'])

## 누락·권한 검토

합성 표시가 없는 레코드를 자동 승인하지 않습니다. 실제 데이터는 별도 사용 허가가 필요합니다.

In [ ]:
bad[3]['instruction'] = ''
bad[4]['synthetic'] = False
display(audit(bad)['issues'])

## 문장 단위 분할의 위험

표현이 다른 동일 사건이 train/test에 나뉘는 의도적인 잘못된 예시입니다.

In [ ]:
naive = {'train': data[::2], 'test': data[1::2]}
display(leakage(naive))
assert len(leakage(naive)[0]['shared_groups']) == 24

## 사례 단위 분할

같은 사건의 모든 표현을 한 분할에 모읍니다. 개발용 dev와 최종 test를 분리합니다.

In [ ]:
splits = split_groups(data, seed=42)
print({k: len(v) for k, v in splits.items()})
assert not leakage(splits)

## 분할 재현성

같은 seed가 같은 분할을 만듭니다. seed 변경과 성능 개선은 다른 주장입니다.

In [ ]:
assert splits == split_groups(data, seed=42)
other = split_groups(data, seed=7)
print('seed 변경 → 같은 분할?', splits == other)

## 문자 길이 점검

문자 수는 토큰 수가 아닙니다. 실제 학습 전 해당 토크나이저로 토큰 길이를 다시 확인해야 합니다.

In [ ]:
lengths = [len(r['instruction'] + r['input'] + r['response']) for r in data]
print('문자 길이 min/median/max:', min(lengths), np.median(lengths), max(lengths))

## 검증 세트 오염의 탐지

분할 이후 데이터가 추가될 때에도 누출 검사를 다시 수행합니다.

In [ ]:
polluted = {k: list(v) for k, v in splits.items()}
polluted['test'].append(splits['train'][0])
assert leakage(polluted)
display(leakage(polluted))

## 대화 템플릿

모델별 역할·종료 토큰은 다릅니다. 아래는 역할 구조 설명용이며 특정 모델 토큰이 아닙니다.

In [ ]:
row = data[0]
chat = [{'role': 'user', 'content': row['instruction'] + '\n' + row['input']}, {'role': 'assistant', 'content': row['response']}]
display(chat)

## 손실 마스킹의 경계

가상 토큰 ID로 입력/답변 경계를 확인합니다. 실제 tokenizer나 trainer 실행이 아닙니다.

In [ ]:
token_ids = [10, 11, 12, 20, 21, 22]
labels = [-100, -100, -100, 20, 21, 22]
assert sum(x != -100 for x in labels) == 3
print('답변 학습 위치:', [i for i, x in enumerate(labels) if x != -100])

## Dataset Card 기록

자료 출처·사용 범위·분할 기준·한계를 명시합니다.

In [ ]:
card = {'source': '수업 합성 자료', 'permission': '교육 실험', 'rows': len(data), 'split': {k: len(v) for k,v in splits.items()}, 'seed': 42, 'limitation': '실제 고객 분포를 대표하지 않음'}
save_report(OUT / 'dataset-card.json', card)
display(card)

## 실행 기록과 기획서 연결

1. 바꾼 변수와 달라진 결과
2. 이 실험으로 확인하지 못한 부분
3. 프로젝트에서 추가로 검증할 가정

출력은 `output/validation/adaptation-labs/`에 저장됩니다. 개인정보나 학생 제출물은 공개 저장소에 넣지 않습니다.